In [ ]:
path_to_dataset = "" # FIXME: Add path to dataset here

In [ ]:
import pandas as pd
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import sys
sys.path.append('../Legacy')

from Legacy_pipeline import LegacyPipeline

csv_path = "Test_dataset.csv"

**Helper functions**

In [ ]:
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union for two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-5)
    return iou

def yolo_to_xyxy(x_center, y_center, w, h, img_w, img_h):
    """Convert YOLO format to [x1, y1, x2, y2]"""
    return [
        int((x_center - w/2) * img_w),
        int((y_center - h/2) * img_h),
        int((x_center + w/2) * img_w),
        int((y_center + h/2) * img_h)
    ]

**Initialisation**

In [ ]:
# Initialize data and pipeline
df = pd.read_csv(csv_path)
pipeline = LegacyPipeline()

**Testing legacy pipeline**

In [ ]:
def evaluate_method(df, pipeline, path_to_dataset, use_adaptive_threshold=False):
    metrics = {
        "border_correct": 0,
        "border_total": 0,        
        "fp_border_but_zero_notches": 0, 
        "fp_border_with_notches": 0,
        "notch_true_positives": 0,
        "notch_false_positives": 0, 
        "notch_false_negatives": 0,
    }
    missed_notches_list = []    
    false_positives_list = []  
    IOU_THRESHOLD = 0.15
    
    print(f"Evaluating Pipeline (Adaptive Threshold: {use_adaptive_threshold})...")
    
    for img_file, group in df.groupby('image_filename'):
        subfolder = img_file[0].upper()
        img_path = os.path.join(path_to_dataset, subfolder, img_file)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
            
        img_h, img_w = img.shape[:2]
        gt_has_border = str(group.iloc[0]['has_border']).lower() in ['true', '1']
        
        # Run Legacy Pipeline
        results = pipeline.process_image(img, use_adaptive_threshold=use_adaptive_threshold)
        predicted_notches = results.get('accepted_notches', [])

        pred_has_border = results['border_present']
        if pred_has_border == gt_has_border:
            metrics["border_correct"] += 1
        elif pred_has_border and not gt_has_border:
            if len(results['accepted_notches']) == 0:
                metrics["fp_border_but_zero_notches"] += 1
            else:
                metrics["fp_border_with_notches"] += 1
                
        metrics["border_total"] += 1
        
        # Extract GT notches 
        gt_notches = group[group['super_category'].str.lower() == 'notch']
        matched_preds = set()
        
        # Check True Positives and False Negatives
        for _, gt_row in gt_notches.iterrows():
            gt_box = yolo_to_xyxy(gt_row['x_center'], gt_row['y_center'], gt_row['width'], gt_row['height'], img_w, img_h)
            
            best_iou = 0
            best_pred_idx = -1
            
            for idx, pred in enumerate(predicted_notches):
                if idx in matched_preds:
                    continue
                iou = calculate_iou(pred['coords'], gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_pred_idx = idx
            
            if best_iou >= IOU_THRESHOLD:
                metrics["notch_true_positives"] += 1
                matched_preds.add(best_pred_idx)
            else:
                metrics["notch_false_negatives"] += 1
                missed_notches_list.append({
                    "image_filename": img_file,
                    "gt_box": gt_box
                })
                
        # Any unmatched predictions are False Positives
        for idx, pred in enumerate(predicted_notches):
            if idx not in matched_preds:
                false_positives_list.append({
                    "image_filename": img_file,
                    "pred_box": pred['coords']
                })
        metrics["notch_false_positives"] += len(predicted_notches) - len(matched_preds)

    # Calculate border accuracy
    b_acc = (metrics["border_correct"] / metrics["border_total"]) * 100 if metrics["border_total"] > 0 else 0

        
    # Calculate Precision & Recall
    tp = metrics["notch_true_positives"]
    fp = metrics["notch_false_positives"]
    fn = metrics["notch_false_negatives"]
    
    precision = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0
    recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    
    print("-" * 40)
    print(f"Results (Adaptive={use_adaptive_threshold}):")
    print(f"1. Border Detection")
    print(f"   Overall Accuracy          : {b_acc:.1f}% ({metrics['border_correct']}/{metrics['border_total']})")
    print(f"   -> FP Borders (0 notches predicted) : {metrics['fp_border_but_zero_notches']}")
    print(f"   -> FP Borders (>0 notches predicted): {metrics['fp_border_with_notches']}")
    print("-" * 40)
    print(f"2. Notch detection")
    print(f"    Precision : {precision:.2f}%")
    print(f"    Recall    : {recall:.2f}%")
    print(f"    TP: {tp} | FP: {fp} | FN: {fn}")
    print("-" * 40 + "\n")
    
    return missed_notches_list, false_positives_list

In [ ]:
# 1. Test Peak Decay Method
missed_peak, fp_peak = evaluate_method(df, pipeline, path_to_dataset, use_adaptive_threshold=False)

# 2. Test Adaptive Thresholding Method
missed_adaptive, fp_adaptive = evaluate_method(df, pipeline, path_to_dataset, use_adaptive_threshold=True)

**Visualisation of mistakes**

In [ ]:
%matplotlib qt 

def interactive_error_viewer(missed_list, fp_list, path_to_dataset, method_name=""):
    errors_by_image = {}
    
    for item in missed_list:
        img = item['image_filename']
        if img not in errors_by_image: errors_by_image[img] = {'missed': [], 'fps': []}
        errors_by_image[img]['missed'].append(item)
        
    for item in fp_list:
        img = item['image_filename']
        if img not in errors_by_image: errors_by_image[img] = {'missed': [], 'fps': []}
        errors_by_image[img]['fps'].append(item)

    image_files = list(errors_by_image.keys())
    
    if not image_files:
        print(f"No errors to visualize for {method_name}!")
        return

    current_idx = 0
    total_images = len(image_files)

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.canvas.manager.set_window_title(f'Legacy Detection Errors - {method_name}')

    def draw_image(idx):
        ax.clear()
        img_file = image_files[idx]
        data = errors_by_image[img_file]
        
        subfolder = img_file[0].upper()
        full_img_path = os.path.join(path_to_dataset, subfolder, img_file)
        img = cv2.imread(full_img_path)
        
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        else:
            ax.text(0.5, 0.5, "Image file not found on disk", ha='center', fontsize=14)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)

        ax.set_title(f"[{idx + 1} / {total_images}] {method_name} | File: {img_file}\n"
                     f"Missed (Orange): {len(data['missed'])} | FPs (Red): {len(data['fps'])}\n"
                     "(Use Left/Right Arrows. Press 'Esc' to close)", 
                     fontsize=12, color='darkred', weight='bold')
        
        # Draw Missed Notches (False Negatives)
        for m in data['missed']:
            x1, y1, x2, y2 = m['gt_box']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='orange', facecolor='none')
            ax.add_patch(rect)
            
        # Draw False Positives
        for fp in data['fps']:
            x1, y1, x2, y2 = fp['pred_box']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='red', facecolor='none', linestyle='--')
            ax.add_patch(rect)
        
        ax.axis('off')
        fig.canvas.draw_idle()

    def on_key_press(event):
        nonlocal current_idx
        if event.key == 'right':
            current_idx = (current_idx + 1) % total_images
            draw_image(current_idx)
        elif event.key == 'left':
            current_idx = (current_idx - 1) % total_images
            draw_image(current_idx)
        elif event.key == 'escape':
            plt.close(fig)

    fig.canvas.mpl_connect('key_press_event', on_key_press)
    draw_image(current_idx)
    plt.tight_layout()
    plt.show()

In [ ]:
interactive_error_viewer(missed_peak, fp_peak, path_to_dataset, "Peak Decay Method")

In [ ]:
interactive_error_viewer(missed_adaptive, fp_adaptive, path_to_dataset, "Adaptive Method")